In [ ]:
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
#import sympy as sy
import pandas as pd
import matplotlib.gridspec as gridspec
from scipy import stats as ss
pd.set_option('display.max_rows', 20)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['lines.markersize'] = 4
plt.rcParams['font.size'] = 14
plt.rcParams['axes.prop_cycle'] = plt.cycler('color', 'brgmyk')


In [ ]:
import h5py

In [ ]:
with h5py.File('C:/Users/polgaste/Desktop/MLS_v6/N2O/MLS-Aura_L2GP-N2O_v06-01-c01_2008d043.he5', 'r') as f:
    print(list(f['/HDFEOS/SWATHS'].keys()))   # should show N2O only, no N2O-640
    q = f['/HDFEOS/SWATHS/N2O/Data Fields/Quality'][:]
    print((q <= 0.8).mean(), (q <= 1.4).mean())

In [ ]:
with h5py.File('C:/Users/polgaste/Desktop/MLS_v6/N2O/MLS-Aura_L2GP-N2O_v06-01-c01_2008d043.he5', 'r') as f:
    print(list(f['/HDFEOS/SWATHS'].keys()))   # should show N2O only, no N2O-640
    c = f['/HDFEOS/SWATHS/N2O/Data Fields/Convergence'][:]
    print((c >= 2.0).mean(), (c >= 1.01).mean())

In [ ]:
with h5py.File('C:/Users/polgaste/Desktop/MLS_v6/N2O/MLS-Aura_L2GP-N2O_v06-01-c01_2008d043.he5', 'r') as f:
    n2o = f['/HDFEOS/SWATHS/N2O/Data Fields/L2gpValue'][:, 7] * 1e9   # 68 hPa
    lat = f['/HDFEOS/SWATHS/N2O/Geolocation Fields/Latitude'][:]
    ok = (c < 1.01)
    dropped = (c >= 1.01) & (c < 2.0)
    print(n2o[ok].mean(), n2o[dropped].mean())
    print(np.abs(lat[ok]).mean(), np.abs(lat[dropped]).mean())

In [ ]:
# read files with monthly average values
data_68 = pd.read_csv('monthly_n2o_68.csv')
data_46 = pd.read_csv('monthly_n2o_46.csv')
data_32 = pd.read_csv('monthly_n2o_32.csv')
data_22 = pd.read_csv('monthly_n2o_22.csv')

In [ ]:
data_68['date'] = pd.to_datetime(data_68['date'])
data_46['date'] = pd.to_datetime(data_46['date'])
data_32['date'] = pd.to_datetime(data_32['date'])
data_22['date'] = pd.to_datetime(data_22['date'])

In [ ]:
o3_68 = pd.read_csv('monthly_o3_68.csv')
o3_46 = pd.read_csv('monthly_o3_46.csv')
o3_32 = pd.read_csv('monthly_o3_32.csv')
o3_22 = pd.read_csv('monthly_o3_22.csv')

In [ ]:
o3_68['date'] = pd.to_datetime(o3_68['date'])
o3_46['date'] = pd.to_datetime(o3_46['date'])
o3_32['date'] = pd.to_datetime(o3_32['date'])
o3_22['date'] = pd.to_datetime(o3_22['date'])

In [ ]:
zone_dict = {
    "bashkortostan": 1,
    "bozeman": 1,
    "brunei": 2,
    "california": 1,
    "catalonia": 1,
    "colombia": 2,
    "estonia": 1,
    "finland": 1,
    "florianopolis": 2,
    "florida": 2,
    "france": 1,
    "french_guiana": 2,
    "huntingdon": 1,
    "iceland_e": 1,
    "iceland_w": 1,
    "khabarovsk": 1,
    "kongo": 2,
    "kyrgyzstan": 1,
    "mexico": 2,
    "morocco": 1,
    "mukhrino": 1,
    "myanmar": 2,
    "nz_n": 3,
    "nz_s": 3,
    "pantanal": 2,
    "quistococha": 2,
    "romania": 1,
    "taiwan": 2,
    "tarapoto": 2,
    "tasmania": 3,
    "tierra_del_fuego": 3,
    "uganda_e": 2,
    "uganda_n": 2,
    "uganda_s": 2,
    "wales": 1
}

In [ ]:
data_68["latitude_zone"] = data_68["location"].map(zone_dict)
data_46["latitude_zone"] = data_46["location"].map(zone_dict)
data_32["latitude_zone"] = data_32["location"].map(zone_dict)
data_22["latitude_zone"] = data_22["location"].map(zone_dict)

In [ ]:
o3_68["latitude_zone"] = o3_68["location"].map(zone_dict)
o3_46["latitude_zone"] = o3_46["location"].map(zone_dict)
o3_32["latitude_zone"] = o3_32["location"].map(zone_dict)
o3_22["latitude_zone"] = o3_22["location"].map(zone_dict)

In [ ]:
data_68 = data_68[data_68['count'] >= 5]
data_46 = data_46[data_46['count'] >= 5]
data_32 = data_32[data_32['count'] >= 5]
data_22 = data_22[data_22['count'] >= 5]

In [ ]:
o3_68 = o3_68[o3_68['count'] >= 5]
o3_46 = o3_46[o3_46['count'] >= 5]
o3_32 = o3_32[o3_32['count'] >= 5]
o3_22 = o3_22[o3_22['count'] >= 5]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats as ss

def plot_monthly_zone_means_side_by_side(
    dfs_n2o: list,
    dfs_o3: list,
    altitudes: list,
    value_col: str = 'mean_concentration',
    date_col: str = 'date',
    zone_col: str = 'latitude_zone',
    confidence: float = 0.95,
    palette: list = None,
    zone_labels: dict = None,
    zone_ci_alpha: dict = None,
    save_path=None,
    save_dpi=300,
    ylabels: dict = None,
    ylims: dict = None
) -> None:
    """
    Plots monthly mean concentrations with 95% CI for latitude zones,
    in a grid with one row per altitude and two columns (N2O on left, O3 on right),
    using fixed y-limits per species.
    """
    # Defaults
    default_palette = ['#4daf4a', '#ff7f00', '#377eb8']
    palette = palette or default_palette
    zone_labels = zone_labels or {1: 'NH', 2: 'Tropics', 3: 'SH'}
    zone_ci_alpha = zone_ci_alpha or {}
    default_alpha = 0.3
    ylabels = ylabels or {
        'N2O': 'Mean N₂O concentration (ppbv)',
        'O3':  'Mean O₃ concentration (ppmv)'
    }
    # default y-limits
    ylims = ylims or {
        'N2O': (0, 350),
        'O3':  (0, 8)
    }

    def prepare(df):
        df = df.copy()
        df[date_col] = pd.to_datetime(df[date_col])
        df['year_month'] = df[date_col].dt.to_period('M').dt.to_timestamp()
        stats_df = (
            df.groupby([zone_col, 'year_month'])[value_col]
              .agg(['mean', 'std', 'count'])
              .reset_index()
        )
        alpha = 1 - confidence
        stats_df['ci_half'] = stats_df.apply(
            lambda r: ss.t.ppf(1 - alpha/2, df=r['count']-1) * r['std'] / (r['count']**0.5)
            if r['count'] > 1 else 0,
            axis=1
        )
        stats_df['ci_lower'] = stats_df['mean'] - stats_df['ci_half']
        stats_df['ci_upper'] = stats_df['mean'] + stats_df['ci_half']
        return stats_df

    stats_n2o = [prepare(df) for df in dfs_n2o]
    stats_o3  = [prepare(df) for df in dfs_o3]

    zones = sorted(
        {z for stats in stats_n2o + stats_o3 for z in stats[zone_col].unique()},
        reverse=True
    )

    n_alt = len(altitudes)
    fig, axes = plt.subplots(n_alt, 2,
                             figsize=(14, 4 * n_alt),
                             sharex=True)

    for row, alt in enumerate(altitudes):
        for col, (stats_list, species) in enumerate(zip((stats_n2o, stats_o3), ('N2O', 'O3'))):
            ax = axes[row, col]
            stats_df = stats_list[row]

            # plot each zone
            for idx, zone in enumerate(zones):
                grp = stats_df[stats_df[zone_col] == zone]
                color = palette[idx % len(palette)]
                alpha_ci = zone_ci_alpha.get(zone, default_alpha)
                ax.fill_between(
                    grp['year_month'], grp['ci_lower'], grp['ci_upper'],
                    color=color, alpha=alpha_ci, zorder=1
                )
                ax.plot(
                    grp['year_month'], grp['mean'],
                    color=color,
                    label=zone_labels.get(zone, f'Zone {zone}'),
                    zorder=2
                )

            # center title = pressure level
            ax.set_title(alt, fontsize=18)

            # y-label and y-limits
            ax.set_ylabel(ylabels[species], fontsize=16)
            ax.set_ylim(ylims[species])

            # grid & x-axis formatting
            ax.grid(True)
            ax.xaxis.set_major_locator(mdates.YearLocator(2))
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
            if row != n_alt - 1:
                ax.tick_params(labelbottom=False)
            else:
                for lbl in ax.get_xticklabels():
                    lbl.set_rotation(45)
                    lbl.set_ha('right')

    # shared legend at bottom
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles[::-1], labels[::-1],
               loc='lower center', ncol=len(zones),
               fontsize=18, bbox_to_anchor=(0.5, 0.01))

    # tighten up spacing
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    if save_path:
        plt.savefig(save_path, dpi=save_dpi)
    plt.show()


In [ ]:
plot_monthly_zone_means_side_by_side(
    dfs_n2o=[data_22, data_32, data_46, data_68],
    dfs_o3 =[o3_22,  o3_32,  o3_46,  o3_68 ],
    altitudes=['22 hPa', '32 hPa', '46 hPa', '68 hPa'],
    value_col='mean_concentration',
    date_col='date',
    zone_col='latitude_zone',
    confidence=0.95,
    palette=None,
    zone_labels={1: 'Northern Hemisphere', 2: 'Tropics', 3: 'Southern Hemisphere'},
    zone_ci_alpha={3: 0.15},
    ylabels={
        'N2O': 'N₂O (ppbv)',
        'O3':  'O₃ (ppmv)'
    },
    save_path = 'lat_zones_v6_4x2.png'
)




In [ ]:
plot_monthly_zone_means_side_by_side(
    dfs_n2o=[data_32, data_46],
    dfs_o3 =[o3_32, o3_46],
    altitudes=['32 hPa', '46 hPa'],
    value_col='mean_concentration',
    date_col='date',
    zone_col='latitude_zone',
    confidence=0.95,
    palette=None,
    zone_labels={1: 'Northern Hemisphere', 2: 'Tropics', 3: 'Southern Hemisphere'},
    zone_ci_alpha={3: 0.15},
    ylabels={
        'N2O': 'N₂O (ppbv)',
        'O3':  'O₃ (ppmv)'
    },
    #save_path = 'lat_zones_4x2.png'
)




In [ ]:
def prepare_monthly_stats(
    df: pd.DataFrame,
    date_col: str = 'date',
    zone_col: str = 'latitude_zone',
    value_col: str = 'mean_concentration',
    confidence: float = 0.95
) -> pd.DataFrame:
    """
    Turn a raw DataFrame into monthly, per-zone means ± CI.
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df['year_month'] = df[date_col].dt.to_period('M').dt.to_timestamp()
    stats = (
        df.groupby([zone_col, 'year_month'])[value_col]
          .agg(['mean', 'std', 'count'])
          .reset_index()
    )
    alpha = 1 - confidence
    stats['ci_half'] = stats.apply(
        lambda r: ss.t.ppf(1 - alpha/2, df=r['count']-1) * r['std'] / (r['count']**0.5)
                  if r['count'] > 1 else 0,
        axis=1
    )
    stats['ci_lower'] = stats['mean'] - stats['ci_half']
    stats['ci_upper'] = stats['mean'] + stats['ci_half']
    return stats

In [ ]:
stats68_n2o = prepare_monthly_stats(df = data_68)
stats46_n2o = prepare_monthly_stats(df = data_46)
stats32_n2o = prepare_monthly_stats(df = data_32)
stats22_n2o = prepare_monthly_stats(df = data_22)

In [ ]:
stats68_o3 = prepare_monthly_stats(df = o3_68)
stats46_o3 = prepare_monthly_stats(df = o3_46)
stats32_o3 = prepare_monthly_stats(df = o3_32)
stats22_o3 = prepare_monthly_stats(df = o3_22)

In [ ]:
import pymannkendall as mk

def mann_kendall_test(stats_df, 
                      date_col: str = 'year_month', 
                      value_col: str = 'mean'):
    """
    Runs the Mann–Kendall test on stats_df[value_col] sorted by date_col.
    Returns the test result object from pymannkendall.
    """
    # make sure it’s sorted
    series = (stats_df
              .sort_values(date_col)
              .set_index(date_col)[value_col])
    return mk.original_test(series)



In [ ]:
for zone, grp in stats68_n2o.groupby('latitude_zone'):
    res = mann_kendall_test(grp)
    print(f"Zone {zone}:")
    print("  trend:    ", res.trend)
    print("  p-value:  ", f"{res.p:.3e}")      # scientific notation if tiny
    print("  S:        ", res.s)              # lowercase s
    print("  Z:        ", res.z)              # normalized statistic
    print("  Tau:      ", res.Tau)            # Kendall’s Tau
    print("  slope:    ", res.slope)          # Sen’s slope estimate
    print()


def report_mann_kendall(
    stats_df,
    zone_col: str    = 'latitude_zone',
    date_col: str    = 'year_month',
    value_col: str   = 'mean'
):
    """
    Runs a Mann–Kendall test on stats_df[value_col] for each unique zone.
    Prints a formatted report per zone and returns a dict {zone: test_result}.
    """
    results = {}
    zones = sorted(stats_df[zone_col].unique())
    for zone in zones:
        df_zone = stats_df[stats_df[zone_col] == zone]
        series = (
            df_zone
            .sort_values(date_col)
            .set_index(date_col)[value_col]
        )
        res = mk.original_test(series)
        results[zone] = res

        # Print formatted report
        print(f"--- Zone {zone} ({value_col!r}) ---")
        print(f"Trend:      {res.trend}")
        print(f"P-value:    {res.p:.3e}")
        print(f"S (stat):   {res.s}")
        print(f"Z:          {res.z:.3f}")
        print(f"Tau:        {res.Tau}")
        print(f"Sen slope:  {res.slope}")
        print()

    return results


In [ ]:
from scipy.stats import theilslopes

def report_mann_kendall(
    stats_df,
    zone_col: str  = 'latitude_zone',
    date_col: str  = 'year_month',
    value_col: str = 'mean'
):
    """
    Mann–Kendall test per zone, plus Theil–Sen slope per decade (absolute and %)
    with a 95% CI, computed against real time in decimal years.
    """
    results = {}
    for zone in sorted(stats_df[zone_col].unique()):
        s = (stats_df[stats_df[zone_col] == zone]
             .sort_values(date_col)
             .set_index(date_col)[value_col])
        res = mk.original_test(s)

        t = s.index.year + (s.index.month - 0.5) / 12          # decimal years, mid-month
        slope, _, lo, hi = theilslopes(s.values, t)            # units per year
        per_dec = slope * 10
        pct_dec = 100 * per_dec / s.mean()

        results[zone] = {'mk': res, 'per_decade': per_dec,
                         'ci_per_decade': (lo * 10, hi * 10), 'pct_per_decade': pct_dec}

        print(f"--- Zone {zone} ({value_col!r}) ---")
        print(f"Trend:          {res.trend}")
        print(f"P-value:        {res.p:.3e}")
        print(f"Tau:            {res.Tau:.3f}")
        print(f"Sen slope:      {per_dec:.3f} per decade  (95% CI {lo*10:.3f} to {hi*10:.3f})")
        print(f"Relative:       {pct_dec:.2f} % per decade")
        print(f"Check (pymk×120): {res.slope*120:.3f}")
        print()
    return results

# N2O Mann-Kendall

In [ ]:
report_mann_kendall(stats68_n2o)

In [ ]:
report_mann_kendall(stats46_n2o)

In [ ]:
report_mann_kendall(stats32_n2o)

In [ ]:
report_mann_kendall(stats22_n2o)

# O3 Mann-Kendall

In [ ]:
report_mann_kendall(stats68_o3)

In [ ]:
report_mann_kendall(stats46_o3)

In [ ]:
report_mann_kendall(stats32_o3)

In [ ]:
report_mann_kendall(stats22_o3)

In [ ]:
np.mean(stats68_n2o['mean'])

In [ ]:
np.mean(stats46_n2o['mean'])

In [ ]:
np.mean(stats32_n2o['mean'])

In [ ]:
np.mean(stats22_n2o['mean'])

In [ ]:
254.85588561398808/np.mean(stats68_n2o['mean'])

In [ ]:
203.7545969508148/np.mean(stats46_n2o['mean'])

In [ ]:
159.82880709730128/np.mean(stats32_n2o['mean'])

In [ ]:
154.55422871567535/np.mean(stats22_n2o['mean'])

In [ ]:
stats68_n2o.groupby('latitude_zone').agg({
    #'std':      ['mean','median','max'],
    'mean':  ['mean']
})

In [ ]:
stats46_n2o.groupby('latitude_zone').agg({
    #'std':      ['mean','median','max'],
    'mean':  ['mean']
})

In [ ]:
stats32_n2o.groupby('latitude_zone').agg({
    #'std':      ['mean','median','max'],
    'mean':  ['mean']
})

In [ ]:
stats22_n2o.groupby('latitude_zone').agg({
    #'std':      ['mean','median','max'],
    'mean':  ['mean']
})

In [ ]:
summary68 = stats68_n2o.groupby('latitude_zone').agg({
    #'std':      ['mean','median','max'],
    'ci_half':  ['mean','median','max']
})
print(summary68)

In [ ]:
summary46 = stats46_n2o.groupby('latitude_zone').agg({
    #'std':      ['mean','median','max'],
    'ci_half':  ['mean','median','max']
})
print(summary46)

In [ ]:
summary32 = stats32_n2o.groupby('latitude_zone').agg({
    #'std':      ['mean','median','max'],
    'ci_half':  ['mean','median','max']
})
print(summary32)

In [ ]:
summary22 = stats22_n2o.groupby('latitude_zone').agg({
    #'std':      ['mean','median','max'],
    'ci_half':  ['mean','median','max']
})
print(summary22)

In [ ]:
stats68_o3.groupby('latitude_zone').agg({
    #'std':      ['mean','median','max'],
    'mean':  ['mean']
})

In [ ]:
stats46_o3.groupby('latitude_zone').agg({
    #'std':      ['mean','median','max'],
    'mean':  ['mean']
})

In [ ]:
stats32_o3.groupby('latitude_zone').agg({
    #'std':      ['mean','median','max'],
    'mean':  ['mean']
})

In [ ]:
stats22_o3.groupby('latitude_zone').agg({
    #'std':      ['mean','median','max'],
    'mean':  ['mean']
})

In [ ]:
summary68_o3 = stats68_o3.groupby('latitude_zone').agg({
    #'std':      ['mean','median','max'],
    'ci_half':  ['mean','median','max']
})
print(summary68_o3)

In [ ]:
summary46_o3 = stats46_o3.groupby('latitude_zone').agg({
    #'std':      ['mean','median','max'],
    'ci_half':  ['mean','median','max']
})
print(summary46_o3)

In [ ]:
summary32_o3 = stats32_o3.groupby('latitude_zone').agg({
    #'std':      ['mean','median','max'],
    'ci_half':  ['mean','median','max']
})
print(summary32_o3)

In [ ]:
summary22_o3 = stats22_o3.groupby('latitude_zone').agg({
    #'std':      ['mean','median','max'],
    'ci_half':  ['mean','median','max']
})
print(summary22_o3)

In [ ]:
means68_n2o = stats68_n2o.groupby('latitude_zone')['mean'].mean()
means46_n2o = stats46_n2o.groupby('latitude_zone')['mean'].mean()
means32_n2o = stats32_n2o.groupby('latitude_zone')['mean'].mean()
means22_n2o = stats22_n2o.groupby('latitude_zone')['mean'].mean()

In [ ]:
summary68['ci_half']['mean']/means68_n2o*100

In [ ]:
summary46['ci_half']['mean']/means46_n2o*100

In [ ]:
summary32['ci_half']['mean']/means32_n2o*100

In [ ]:
summary22['ci_half']['mean']/means22_n2o*100

In [ ]:
means68_o3 = stats68_o3.groupby('latitude_zone')['mean'].mean()
means46_o3 = stats46_o3.groupby('latitude_zone')['mean'].mean()
means32_o3 = stats32_o3.groupby('latitude_zone')['mean'].mean()
means22_o3 = stats22_o3.groupby('latitude_zone')['mean'].mean()

In [ ]:
summary68_o3['ci_half']['mean']/means68_o3*100

In [ ]:
summary46_o3['ci_half']['mean']/means46_o3*100

In [ ]:
summary32_o3['ci_half']['mean']/means32_o3*100

In [ ]:
summary22_o3['ci_half']['mean']/means22_o3*100